# Holder concentration — RWA Flow Tracker

Computes top-10 share and Gini coefficient per asset from the DuckDB tables
loaded by `pipeline/load_duckdb.py`.

**Interview-defense notes**
- Top-10 share = sum of 10 largest balances / total supply held. Simple,
  interpretable, standard in tokenomics reporting.
- Gini coefficient: 0 = perfectly equal distribution, 1 = one wallet holds
  everything. Computed on the sorted balance distribution as
  `G = (2 * sum(i * x_i)) / (n * sum(x_i)) - (n + 1) / n` where `i` is the
  1-based rank of each balance `x_i`.
- Caveat to say out loud: balances come from Dune's `balances_ethereum.latest`,
  which counts contract-held balances too; BUIDL additionally rebases, so its
  'supply' grows without Transfer events.

Run `pipeline/dune_pull.py` then `pipeline/load_duckdb.py` first so that
`data/rwa.duckdb` exists.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px

DB = "data/rwa.duckdb"
con = duckdb.connect(DB, read_only=True)

# Which raw tables did the loader create?
con.execute("SHOW TABLES").fetchall()

## Load top-holder tables

The loader stores every CSV under the `raw` schema with all-VARCHAR
columns. Each `04_top_holders` / `07_concentration` Dune query produces a
table like `raw.<query_name>`. Adjust the name pattern below to whatever
you named your saved Dune queries in `pipeline/config.yaml`.

In [ ]:
def load_holder_table(pattern: str) -> pd.DataFrame:
    """Find raw tables matching `pattern`, cast balance columns, return df."""
    names = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
    matches = [n for n in names if pattern in n]
    if not matches:
        print(f"No table matching '{pattern}'. Available: {names}")
        return pd.DataFrame()
    tbl = f"raw.{matches[0]}"
    df = con.execute(f"""
        SELECT
            lower(CAST(address AS VARCHAR)) AS address,
            TRY_CAST(REPLACE(balance, ',', '') AS DOUBLE) AS balance,
            TRY_CAST(REPLACE(balance_usd, ',', '') AS DOUBLE) AS balance_usd
        FROM {tbl}
    """).df()
    df = df[df.balance > 0].sort_values("balance", ascending=False)
    print(f"{tbl}: {len(df)} holders, top balance {df.balance.iloc[0]:,.2f}")
    return df

holders = {
    asset: load_holder_table(asset)
    for asset in ["buidl", "ousg", "usdy"]
}
holders = {k: v for k, v in holders.items() if not v.empty}
holders.keys()

In [ ]:
def concentration_stats(df: pd.DataFrame) -> dict:
    """Top-10 share and Gini for one asset's holder balances."""
    x = df["balance"].to_numpy()
    n = len(x)
    total = x.sum()
    top10 = x[:10].sum()
    # Gini via rank formula; x sorted desc -> reverse to ascending ranks
    xs = x[::-1]
    i = np.arange(1, n + 1)
    gini = (2 * (i * xs).sum()) / (n * xs.sum()) - (n + 1) / n
    return {
        "holders": n,
        "total_tokens": total,
        "top10_share_pct": 100 * top10 / total,
        "gini": gini,
    }

import numpy as np
stats = pd.DataFrame({k: concentration_stats(v) for k, v in holders.items()}).T
stats.round(4)

In [ ]:
# Chart: top-10 share per asset
fig = px.bar(
    stats.reset_index().rename(columns={"index": "asset"}),
    x="asset", y="top10_share_pct",
    title="Top-10 holder share of supply (%)",
)
fig.show()

# Chart: holder count vs top-10 share (concentration vs breadth)
fig2 = px.scatter(
    stats.reset_index().rename(columns={"index": "asset"}),
    x="holders", y="top10_share_pct", size="gini",
    hover_name="asset", title="Holder count vs concentration",
)
fig2.show()

## Next steps (Phase 2)

1. Take the top-20 addresses per asset (`holders[asset].head(20)`) and label
   each in `docs/holder_labels.md` — exchange / issuer / fund / DeFi /
   unknown — recording the label source (Nansen, Arkham, Etherscan, manual).
2. Save the stats table to `docs/` for the write-up; every number must trace
   back to a `sql/` file or this notebook.